# 03 — DOCX Processing

**FinRAG learning series · Notebook 3 of 14**

## Learning objectives
1. Understand the structure of a `.docx` file (a ZIP of XML parts) and why that matters for security.
2. Read paragraphs, headings and tables with **python-docx**, in document order.
3. Preserve the **heading path** (section) and table/row position of every piece of content.
4. Run FinRAG's production `DocxParser` and extract financial values with citations such as *"section: … table 1 row 6"*.

## Prerequisites
Notebooks 01–02. A Word document with a financial table in your documents folder. Saved outputs use a `.docx` that was
*transcribed* from a public report's statement page by `backend/tests/fixtures/build_derived_fixtures.py` (all values copied
verbatim; provenance paragraph included in the file).

## Concept: what is inside a .docx
A `.docx` is a ZIP archive (Office Open XML). The main text is `word/document.xml`; its `<w:body>` holds paragraphs (`<w:p>`)
and tables (`<w:tbl>`) **in reading order**. Headings are just paragraphs with a style such as *Heading 1*.

Because it is a ZIP, an upload could be a *zip bomb* or contain a macro project — FinRAG's validator checks for both
(`backend/app/document_processing/validation.py`).

### FinRAG's DOCX workflow
```
body children in order ─► paragraph? style "Heading N" → update heading path
                        │            otherwise → paragraph element (location: heading path, paragraph index)
                        └► table     → header row + table rows (location: heading path, table index, row)
                                       numeric cells keep their column header (period)
```

In [1]:
# Setup: make the production backend importable and keep secrets out of outputs
import sys, os
sys.path.insert(0, os.path.abspath("."))      # run notebooks from the notebooks/ folder
import finrag_nb as nb                          # helper: paths, safe settings, DB session
print("Backend package:", nb.BACKEND_DIR)
print("Documents folder:", nb.DOCS_DIR)

Backend package: /home/user/Ready_To_Deploy/backend
Documents folder: /opt/finrag_testdocs


## 1. Look inside the ZIP container

In [2]:
import zipfile
docx_path = nb.doc_path("derived/derived_p58.docx")
with zipfile.ZipFile(docx_path) as z:
    for info in z.infolist()[:12]:
        print(f"{info.file_size:>8} bytes  {info.filename}")

    1738 bytes  [Content_Types].xml
     734 bytes  _rels/.rels
     721 bytes  docProps/core.xml
    1132 bytes  docProps/app.xml
   12037 bytes  word/document.xml
    1227 bytes  word/_rels/document.xml.rels
  349458 bytes  word/styles.xml
  438131 bytes  word/stylesWithEffects.xml
    2535 bytes  word/settings.xml
     438 bytes  word/webSettings.xml
    2811 bytes  word/fontTable.xml
   10939 bytes  word/theme/theme1.xml


In [3]:
from app.document_processing.validation import validate_upload
v = validate_upload(docx_path.name, docx_path.read_bytes())
print("validated as:", v.file_format, "| warnings:", v.warnings)

validated as: docx | warnings: []


## 2. Read the document with python-docx, in body order

In [4]:
import docx
from docx.table import Table
from docx.text.paragraph import Paragraph

d = docx.Document(docx_path)
print("core properties:", {k: getattr(d.core_properties, k) for k in ("title", "author", "created") if getattr(d.core_properties, k)})
for child in d.element.body.iterchildren():
    tag = child.tag.rsplit("}", 1)[-1]
    if tag == "p":
        p = Paragraph(child, d)
        if p.text.strip():
            print(f"[paragraph style={p.style.name!r}] {nb.shorten(p.text, 100)}")
    elif tag == "tbl":
        t = Table(child, d)
        print(f"[table] {len(t.rows)} rows x {len(t.columns)} columns; header = {[c.text for c in t.rows[0].cells]}")

core properties: {'author': 'python-docx', 'created': datetime.datetime(2013, 12, 23, 23, 15, tzinfo=datetime.timezone.utc)}
[paragraph style='Heading 1'] Statement of Profit and Loss (C in crore)
[paragraph style='Normal'] Transcribed from RIL-Integrated-Annual-Report-2024-25.pdf (sha256 bb541a68a0440fd4…), PDF page 58, …
[paragraph style='Normal'] (C in crore)
[paragraph style='Normal'] (C in crore)
[table] 29 rows x 3 columns; header = ['Particulars', '2024-25', '2023-24']


**Look at the core properties:** author *python-docx*, created *2013*. They come from the default template of the library that
wrote this transcribed file — not from a real author. Document metadata can be wrong or stale, which is why FinRAG treats file
properties as only one weak signal when detecting the organisation and never as proof.

## 3. Tables: rows, cells and merged cells
python-docx returns the **same cell object repeatedly** for horizontally merged cells. FinRAG collapses consecutive duplicates
(`_unique_cells`) so a merged label is not read twice.

In [5]:
from app.document_processing.parsers.docx_parser import _unique_cells
t = d.tables[0]
for r in t.rows[:6]:
    print(_unique_cells(r))

['Particulars', '2024-25', '2023-24']
['Value of Sales', '5,38,564', '5,33,566']
['Income from Services', '18,599', '41,390']
['Value of Sales & Services (Revenue)', '5,57,163', '5,74,956']
['Less: GST Recovered', '24,371', '27,014']
['Revenue from Operations', '5,32,792', '5,47,942']


## 4. The production parser

In [6]:
from collections import Counter
from app.document_processing.parsers.docx_parser import DocxParser

parsed = DocxParser().parse(str(docx_path))
print(Counter(e.kind for e in parsed.elements), "| tables:", len(parsed.tables), "| warnings:", parsed.warnings)
for e in parsed.elements[:3]:
    print(e.kind, "|", nb.shorten(e.text, 90), "|", e.location.describe())

Counter({'table_row': 28, 'paragraph': 3, 'heading': 1}) | tables: 1 | warnings: []
heading | Statement of Profit and Loss (C in crore) | section: Statement of Profit and Loss (C in crore) paragraph 1
paragraph | Transcribed from RIL-Integrated-Annual-Report-2024-25.pdf (sha256 bb541a68a0440fd4…), PDF… | section: Statement of Profit and Loss (C in crore) paragraph 2
paragraph | (C in crore) | section: Statement of Profit and Loss (C in crore) paragraph 3


In [7]:
rows = [e for e in parsed.elements if e.kind == "table_row"]
for e in rows[:6]:
    print(f"{e.location.describe():<55} | {nb.shorten(e.text, 80)}")

section: Statement of Profit and Loss (C in crore) table 1 row 2 | Value of Sales | 2024-25: 5,38,564 | 2023-24: 5,33,566
section: Statement of Profit and Loss (C in crore) table 1 row 3 | Income from Services | 2024-25: 18,599 | 2023-24: 41,390
section: Statement of Profit and Loss (C in crore) table 1 row 4 | Value of Sales & Services (Revenue) | 2024-25: 5,57,163 | 2023-24: 5,74,956
section: Statement of Profit and Loss (C in crore) table 1 row 5 | Less: GST Recovered | 2024-25: 24,371 | 2023-24: 27,014
section: Statement of Profit and Loss (C in crore) table 1 row 6 | Revenue from Operations | 2024-25: 5,32,792 | 2023-24: 5,47,942
section: Statement of Profit and Loss (C in crore) table 1 row 7 | Other Income | 2024-25: 16,094 | 2023-24: 12,128


## 5. From rows to financial values with source references
The same metric extractor used for PDFs works on Word tables. Each value's location is the table row, and its column header is
the period.

In [8]:
import pandas as pd
from app.financial_analytics.metric_extractor import extract_metrics

ms = extract_metrics(parsed.elements, "docx", doc_currency=None, doc_unit=None)
df = pd.DataFrame([{"metric": m.metric_key, "printed": m.raw_value, "period": m.period_label, "unit": m.unit,
                    "source": m.location.describe()} for m in ms if m.metric_key != "line_item"])
print(len(ms), "values extracted;", len(df), "map to canonical metrics")
df.head(12)

56 values extracted; 28 map to canonical metrics


,metric,printed,period,unit,source
0,revenue_from_operations,"5,32,792",2024-25,crore,section: Statement of Profit and Loss (C in cr...
1,revenue_from_operations,"5,47,942",2023-24,crore,section: Statement of Profit and Loss (C in cr...
2,other_income,"16,094",2024-25,crore,section: Statement of Profit and Loss (C in cr...
3,other_income,"12,128",2023-24,crore,section: Statement of Profit and Loss (C in cr...
4,total_income,"5,48,886",2024-25,crore,section: Statement of Profit and Loss (C in cr...
5,total_income,"5,60,070",2023-24,crore,section: Statement of Profit and Loss (C in cr...
6,cost_of_materials,"3,84,021",2024-25,crore,section: Statement of Profit and Loss (C in cr...
7,cost_of_materials,"3,76,418",2023-24,crore,section: Statement of Profit and Loss (C in cr...
8,employee_benefits_expense,"8,690",2024-25,crore,section: Statement of Profit and Loss (C in cr...
9,employee_benefits_expense,"7,807",2023-24,crore,section: Statement of Profit and Loss (C in cr...


## Interpretation
* Body order matters: the heading paragraphs come before the table, so every table row inherits the section path.
* Units: the title paragraphs of this file include "(C in crore)", which FinRAG reads as the unit *crore* for the table — check the
  `unit` column above.
* The citation text (*section … table 1 row N*) is what the chat shows for Word sources.

## Common errors and troubleshooting
| Symptom | Cause | Fix |
|---|---|---|
| `PackageNotFoundError` | Not a real .docx (e.g. renamed .doc) | Save as .docx; FinRAG's validator rejects mismatched files |
| Headings not detected | Author used bold text instead of Heading styles | Content is still extracted; sections fall back to the nearest heading |
| Duplicated cell text | Merged cells | Use `_unique_cells` logic |
| Tables inside text boxes missing | python-docx does not read text boxes/shapes | Convert to normal tables |
| Upload rejected "Macro-enabled content" | `vbaProject.bin` present | Save without macros |

## Practical exercises
1. Upload one of your own Word financial reports and compare the heading paths FinRAG records with the document outline.
2. Add a second table to a copy of a document under a new heading; confirm rows get the new section in their location.
3. Find a merged header cell in your document and print the raw `row.cells` versus `_unique_cells(row)`.

## Summary
You opened a DOCX as a ZIP, walked its body in order, handled merged cells, and used the production parser + metric extractor to
produce values cited by section, table and row.

## Further learning
* python-docx: https://python-docx.readthedocs.io/
* ECMA-376 Office Open XML overview: https://ecma-international.org/publications-and-standards/standards/ecma-376/
* FinRAG source: `backend/app/document_processing/parsers/docx_parser.py`
* Next: **04 — Excel and CSV processing**.